In [17]:
%matplotlib widget

from pathlib import Path
import runpy

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display


# Files
ROOT = Path.cwd()
CHECKPOINT = ROOT / "velocity.pt"
CASE_ROOT = ROOT / "cutoff_cases"

# Physical inputs
D = 0.15            # Pipe diameter [m]
PIPE_LENGTH = 30.0  # Physical pipe length [m]
U = 0.007           # Mean inlet velocity [m/s]
NU = 1e-6           # Kinematic viscosity [m²/s]

# Display settings
PLOT_LENGTH = 25.0  # Displayed axial range [m]
MAX_TIME = 40000    # Displayed time range [s]
START_X = 2.0
START_TIME = 1000
X_STEP = 0.25
TIME_STEP = 50

script = next(
    (
        ROOT / name
        for name in ("pipe_ml_all_lr.py", "pipe_ml_all_lr(1).py")
        if (ROOT / name).is_file()
    ),
    None,
)

if script is None:
    raise FileNotFoundError(f"Put pipe_ml_all_lr.py in {ROOT}")
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Checkpoint missing: {CHECKPOINT}")

code = runpy.run_path(str(script))
code["torch"].set_num_threads(code["CPU_THREADS"])
model, _ = code["load_model"](CHECKPOINT)
raw_velocity = code["raw_velocity"]

training_folders = sorted(path.parent for path in CASE_ROOT.glob("*/case.json"))

if not training_folders:
    raise FileNotFoundError(f"No exported cutoff cases found in {CASE_ROOT}")
predictor = code["CutoffPredictor"](training_folders, model)
le, ts = map(float, predictor.cutoffs(D, PIPE_LENGTH, U, NU))

lower = model.lower.detach().cpu().numpy()
upper = model.upper.detach().cpu().numpy()
reynolds = U * D / NU
x_min, x_max = float(lower[1]) * D, float(upper[1]) * D
t_min, t_max = float(lower[3]) * D / U, float(upper[3]) * D / U

if not lower[0] - 1e-6 <= reynolds <= upper[0] + 1e-6:
    raise ValueError(f"Re = {reynolds:g} is outside the checkpoint's range.")
if not x_min - 1e-6 <= le <= x_max + 1e-6:
    raise ValueError(
        f"Predicted Le = {le:.3f} m is outside the checkpoint's "
        f"axial range of {x_min:.3f}–{x_max:.3f} m."
    )
if not t_min - 1e-6 <= ts <= t_max + 1e-6:
    raise ValueError(
        f"Predicted ts = {ts:.1f} s is outside the checkpoint's "
        f"time range of {t_min:.1f}–{t_max:.1f} s."
    )

summary = widgets.HTML(
    f"<b>Predicted Lₑ</b> {le:.3f} m &nbsp; | &nbsp; "
    f"<b>Predicted tₛ</b> {ts:.1f} s"
)

dx = X_STEP / 2
x_positions = np.arange(int(np.ceil(PLOT_LENGTH / dx)) + 1) * dx
y_positions = np.linspace(-D / 2, D / 2, 101)
dy = y_positions[1] - y_positions[0]

x_grid, y_grid = np.meshgrid(np.minimum(x_positions, PLOT_LENGTH), y_positions)
r_grid = np.abs(y_grid)
steady_field = None


def calculate_field(time):
    global steady_field

    steady = time >= ts
    if steady and steady_field is not None:
        return steady_field

    t_eval = min(float(time), ts)
    x_eval = np.minimum(x_grid, le) if steady else x_grid
    field = np.full(x_grid.shape, np.nan)

    if t_min - 1e-6 <= t_eval <= t_max + 1e-6:
        valid = (x_eval >= x_min - 1e-6) & (x_eval <= x_max + 1e-6)
        if np.any(valid):
            velocity = raw_velocity(
                model, U, D, NU,
                np.clip(x_eval[valid], x_min, x_max),
                r_grid[valid],
                float(np.clip(t_eval, t_min, t_max)),
            )
            field[valid] = np.maximum(velocity, 0.0)

    if steady:
        steady_field = field

    return field


x_slider = widgets.FloatSlider(
    value=min(START_X, PLOT_LENGTH), min=0, max=PLOT_LENGTH, step=X_STEP,
    description="x [m]", readout_format=".2f", continuous_update=True,
    layout=widgets.Layout(width="95%"),
)
time_slider = widgets.IntSlider(
    value=min(START_TIME, MAX_TIME), min=0, max=MAX_TIME, step=TIME_STEP,
    description="Time [s]", continuous_update=False,
    layout=widgets.Layout(width="80%"),
)
play = widgets.Play(
    value=time_slider.value, min=0, max=MAX_TIME, step=TIME_STEP,
    interval=500,
)
play_link = widgets.jslink((play, "value"), (time_slider, "value"))
status = widgets.HTML()

current_field = calculate_field(time_slider.value)
cmap = plt.get_cmap("turbo").with_extremes(bad="#d9d9d9")

with plt.ioff():
    fig, (ax_map, ax_profile) = plt.subplots(
        1, 2, figsize=(10, 4), width_ratios=(2.2, 1),
        layout="constrained",
    )

    heatmap = ax_map.imshow(
        current_field,
        origin="lower", aspect="auto", interpolation="nearest",
        extent=(
            -dx / 2, x_positions[-1] + dx / 2,
            y_positions[0] - dy / 2, y_positions[-1] + dy / 2,
        ),
        cmap=cmap, vmin=0, vmax=2 * U,
    )
    ax_map.set(
        xlim=(0, PLOT_LENGTH), ylim=(-D / 2, D / 2),
        xlabel="Axial position x [m]", ylabel="Across pipe y [m]",
    )
    map_title = ax_map.set_title("")
    section_line = ax_map.axvline(x_slider.value, color="white", linewidth=1.8)
    fig.colorbar(heatmap, ax=ax_map, label="Axial velocity [m/s]", extend="max")

    profile_line, = ax_profile.plot(
        y_positions, np.full(y_positions.shape, np.nan),
        linewidth=2, label="ML prediction",
    )
    analytical_profile = 2 * U * (1 - (y_positions / (D / 2)) ** 2)
    ax_profile.plot(
        y_positions, analytical_profile,
        color="black", linestyle="--", linewidth=1.5,
        label="Fully developed reference",
    )
    ax_profile.set(
        xlim=(-D / 2, D / 2), ylim=(0, 2.5 * U),
        xlabel="Across pipe y [m]", ylabel="Axial velocity [m/s]",
    )
    ax_profile.set_xticks(np.linspace(-D / 2, D / 2, 5))
    profile_title = ax_profile.set_title("")
    ax_profile.grid(alpha=0.25)
    ax_profile.legend(loc="lower center", fontsize=8)


def update_section(change=None):
    x = float(x_slider.value)
    column = min(int(np.floor(x / dx + 0.5)), len(x_positions) - 1)

    section_line.set_xdata([x, x])
    profile_line.set_ydata(current_field[:, column])
    profile_title.set_text(f"Profile at x = {x:.2f} m")
    fig.canvas.draw_idle()


def update_time(change=None):
    global current_field

    time = time_slider.value
    current_field = calculate_field(time)
    heatmap.set_data(current_field)
    map_title.set_text(f"Axial velocity at t = {time} s")

    peak = float(np.max(current_field, where=np.isfinite(current_field), initial=0))
    ax_profile.set_ylim(0, max(2.5 * U, 1.05 * peak))

    if time >= ts:
        status.value = (
            f"Using the steady field at tₛ = {ts:.1f} s. "
            f"Downstream of Lₑ = {le:.3f} m, the profile stays constant."
        )
    else:
        status.value = "Transient field. Gray areas are outside checkpoint coverage."

    update_section()


x_slider.observe(update_section, names="value")
time_slider.observe(update_time, names="value")
update_time()

display(widgets.VBox([
    summary,
    x_slider,
    widgets.HBox([play, time_slider]),
    fig.canvas,
    status,
]))